# QLoRA adapter — citation and abstention tuning

The untuned 1.5B baseline is committed in `results/answers_baseline.md`: format 0.093,
grounded answer 0.0, 38/43 forced abstentions. This notebook trains a QLoRA adapter on
synthetic, extractive examples from the **four SFT pages only** and re-runs the same
43-question test split.

- Public repo, no token.
- The three held-out test pages never appear in training contexts.
- The adapter is saved locally and checkpointed to Drive; it is not pushed to the Hub.


In [7]:
REPO_URL = "https://github.com/titidintsako-bit/transformers-docs-copilot.git"
EPOCHS = 2
MAX_SEQ_LENGTH = 2560
LIMIT = 0              # debug: cap train/val examples; 0 uses the full SFT split
print("repo:", REPO_URL)
print("epochs:", EPOCHS, "| max_seq_length:", MAX_SEQ_LENGTH, "| limit:", LIMIT)


repo: https://github.com/titidintsako-bit/transformers-docs-copilot.git
epochs: 2 | max_seq_length: 2560 | limit: 0


## 1. Preflight

Checks the GPU, free disk, Python and that the public repo is reachable.


In [8]:
import shutil
import subprocess

import torch

assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
props = torch.cuda.get_device_properties(0)
free_gb = shutil.disk_usage("/content").free / 1e9
assert free_gb > 8, f"Only {free_gb:.1f} GB free in /content; free space and retry"

print(f"python: {__import__('sys').version.split()[0]}")
print(f"torch: {torch.__version__} | {props.name} | {props.total_memory/1e9:.1f} GB VRAM")
print(f"free disk: {free_gb:.1f} GB")
head = subprocess.run(["git", "ls-remote", REPO_URL, "HEAD"], capture_output=True, text=True, check=True)
print("repo head:", head.stdout.split()[0][:12])


python: 3.13.16
torch: 2.14.1+cu130 | Tesla T4 | 15.6 GB VRAM
free disk: 117.3 GB
repo head: 5ada5160cf65


## 2. Get the project

Clones the public repo, or pulls the latest commit. A dirty/partial clone is re-cloned.


In [9]:
import os
import shutil
import subprocess
from pathlib import Path

PROJECT_DIR = Path("/content/transformers-docs-copilot")
if PROJECT_DIR.exists() and not (PROJECT_DIR / "pyproject.toml").exists():
    shutil.rmtree(PROJECT_DIR)
if PROJECT_DIR.exists():
    pull = subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"],
        capture_output=True, text=True,
    )
    if pull.returncode != 0:
        print("Repo update failed; re-cloning.")
        print(pull.stderr[-400:])
        shutil.rmtree(PROJECT_DIR)
if not PROJECT_DIR.exists():
    clone = subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)],
        capture_output=True, text=True,
    )
    if clone.returncode != 0:
        raise SystemExit("git clone failed:\n" + clone.stderr[-1200:])
os.chdir(PROJECT_DIR)
print("project:", Path.cwd())
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())


project: /content/transformers-docs-copilot
5ada516 Fix the QLoRA TrainingArguments for transformers 5.24


## 3. Install the training runtime

Adds PEFT to the pinned inference stack. It does not replace Colab's PyTorch.


In [10]:
import subprocess
import sys

packages = [
    "PyYAML==6.0.2",
    "transformers==5.24.0",
    "accelerate==1.16.0",
    "safetensors==0.8.0",
    "huggingface-hub==2.4.0",
    "bitsandbytes==0.50.2",
    "peft==0.22.0",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

import bitsandbytes
import peft
import transformers

print("transformers", transformers.__version__)
print("peft", peft.__version__)
print("bitsandbytes", bitsandbytes.__version__)


transformers 5.24.0
peft 0.22.0
bitsandbytes 0.50.2


## 4. Build the SFT split

Synthetic extractive targets from the four SFT pages, plus 55 unrelated abstention
examples. The held-out pages are excluded from the training retriever itself.


In [11]:
import json
import subprocess
import sys
from pathlib import Path

subprocess.run([sys.executable, "-m", "src.fetch_docs", "--check"], check=True)
subprocess.run([sys.executable, "-m", "src.index"], check=True)
subprocess.run([sys.executable, "-m", "src.build_sft"], check=True)

manifest = json.loads(Path("data/sft/manifest.json").read_text())
print(json.dumps(manifest, indent=2))


{
  "seed": 42,
  "source": "data/chunks.jsonl",
  "sft_chunks": [
    "quicktour:_intro:0",
    "quicktour:set-up:0",
    "quicktour:agent-skills:0",
    "quicktour:pretrained-models:0",
    "quicktour:pretrained-models:1",
    "quicktour:pipeline:0",
    "quicktour:pipeline:1",
    "quicktour:trainer:0",
    "quicktour:trainer:1",
    "quicktour:next-steps:0",
    "models:_intro:0",
    "models:models-and-configurations:0",
    "models:model-classes:0",
    "models:model-classes:1",
    "models:large-models:0",
    "models:sharded-checkpoints:0",
    "models:big-model-inference:0",
    "models:disk-offloading:0",
    "models:model-data-type:0",
    "models:custom-models:0",
    "llm_tutorial:_intro:0",
    "llm_tutorial:default-generate:0",
    "llm_tutorial:default-generate:1",
    "llm_tutorial:generation-configuration:0",
    "llm_tutorial:saving:0",
    "llm_tutorial:common-options:0",
    "llm_tutorial:pitfalls:0",
    "llm_tutorial:output-length:0",
    "llm_tutorial:decoding-s

## 5. Train the adapter

Only assistant answer tokens are supervised; prompt tokens are masked. On a T4 this is
roughly 10–20 minutes for the full split.


In [12]:
import subprocess
import sys

cmd = [
    sys.executable, "-m", "src.train_qlora",
    "--train", "data/sft/train.jsonl",
    "--val", "data/sft/val.jsonl",
    "--output-dir", "artifacts/adapter",
    "--epochs", str(EPOCHS),
    "--max-seq-length", str(MAX_SEQ_LENGTH),
]
if LIMIT:
    cmd += ["--limit", str(LIMIT)]
print(" ".join(cmd))
subprocess.run(cmd, check=True)


/usr/bin/python3 -m src.train_qlora --train data/sft/train.jsonl --val data/sft/val.jsonl --output-dir artifacts/adapter --epochs 2 --max-seq-length 2560


CompletedProcess(args=['/usr/bin/python3', '-m', 'src.train_qlora', '--train', 'data/sft/train.jsonl', '--val', 'data/sft/val.jsonl', '--output-dir', 'artifacts/adapter', '--epochs', '2', '--max-seq-length', '2560'], returncode=0)

## 6. Evaluate on the same test split

`behavior_accuracy` still requires the model itself to choose answer vs abstain. A forced
abstention is a failure even though the delivered output is safe.


In [13]:
import subprocess
import sys

cmd = [
    sys.executable, "-m", "src.eval_answers",
    "--split", "test",
    "--model", "base",
    "--quantize", "on",
    "--adapter", "artifacts/adapter",
    "--out-prefix", "results/answers_tuned",
    "--labels", "results/label_sheet_tuned.csv",
]
print(" ".join(cmd))
subprocess.run(cmd, check=True)


/usr/bin/python3 -m src.eval_answers --split test --model base --quantize on --adapter artifacts/adapter --out-prefix results/answers_tuned --labels results/label_sheet_tuned.csv


CompletedProcess(args=['/usr/bin/python3', '-m', 'src.eval_answers', '--split', 'test', '--model', 'base', '--quantize', 'on', '--adapter', 'artifacts/adapter', '--out-prefix', 'results/answers_tuned', '--labels', 'results/label_sheet_tuned.csv'], returncode=0)

In [14]:
import json
from pathlib import Path
from IPython.display import Markdown, display

payload = json.loads(Path("results/answers_tuned.json").read_text())
print(json.dumps(payload["overall"], indent=2))
display(Markdown(Path("results/answers_tuned.md").read_text()))


{
  "n": 43,
  "behavior_accuracy": 0.93,
  "answered_rate": 0.628,
  "abstained_rate": 0.372,
  "model_abstain_rate_on_negative": 0.933,
  "delivered_abstention_rate_on_negative": 0.933,
  "over_answer_rate_on_negative": 0.067,
  "parser_save_rate_on_negative": 0.0,
  "false_abstain_rate_on_answerable": 0.071,
  "grounded_answer_rate": 0.821,
  "citation_hit_rate": 0.821,
  "page_citation_hit_rate": 0.857,
  "mean_citation_precision": 0.852,
  "format_rate": 1.0,
  "retrieval_hit_rate": 0.964,
  "repairs": 0,
  "forced_abstain": 0,
  "mean_latency_s": 6.197,
  "median_latency_s": 5.36
}


# Grounded answer report (tuned adapter)

- Docs: `huggingface/transformers` @ `v5.24.0` (`eeee11099135`)
- Model: `Qwen/Qwen2.5-1.5B-Instruct` (4-bit NF4), greedy decoding, max_new_tokens=350
- Split: **test**, n=43, k=5 passages
- Contract: `Answer:` plus inline `[n]` citations, canonical `Sources:` rendered by the harness, or exactly `Insufficient evidence in the retrieved documentation.`
- Automated only: format, abstention, citation validity. Support/factual correctness: see the label sheet.

## Overall

| metric | value |
|---|---|
| n | 43 |
| behaviour accuracy | 0.93 |
| answered | 0.628 |
| grounded answer (answerable) | 0.821 |
| citation hits a gold passage (answerable) | 0.821 |
| citation hits a gold page (answerable) | 0.857 |
| mean citation precision (answered) | 0.852 |
| model abstains on negative questions | 0.933 |
| delivered abstention on negative questions | 0.933 |
| delivered answer on negative questions | 0.067 |
| parser saves on negative questions | 0.0 |
| false abstain (answerable) | 0.071 |
| exact format | 1.0 |
| section retrieval hit@k (answerable) | 0.964 |
| citation repairs | 0 |
| forced abstentions | 0 |
| median latency (s) | 5.36 |

## By category

| category | n | behaviour | answered | grounded | format | citation precision |
|---|---|---|---|---|---|---|
| answerable | 28 | 0.929 | 0.929 | 0.821 | 1.0 | 0.885 |
| parametric | 5 | 1.0 | 0.0 | None | 1.0 | None |
| unanswerable | 10 | 0.9 | 0.1 | None | 1.0 | 0.0 |

## By topic

| topic | n | behaviour | answered | citation hit | format |
|---|---|---|---|---|---|
| fine-tuning | 18 | 0.889 | 0.889 | 0.778 | 1.0 |
| generation | 10 | 1.0 | 1.0 | 0.9 | 1.0 |
| out-of-scope | 15 | 0.933 | 0.067 | None | 1.0 |

## Over-answered negative questions

- **un-10** (unanswerable): What is the default value of max_shard_size in save_pretrained?
  - answer: Answer: Parameters load in parallel and peak memory only depends on model size [1]. Use `max_shard_size` in `~PreTrainedModel.save_pretrained` to set the maximum checkpoint size before sharding [1].

Sources:
[1] Loading models - https://gi

## Parser saves on negative questions (model tried to answer)

None.

## False abstentions on answerable questions

- **lm-05** (fine-tuning): Why does the guide flatten the ELI5 dataset before tokenizing?
  - reason: None, retrieved: tasks_language_modeling#load-eli5-dataset
- **lm-08** (fine-tuning): How does the guide run inference after fine-tuning the causal language model?
  - reason: None, retrieved: training#None

## Answered but no cited gold passage

- **gs-01** (generation): cited [1], gold generation_strategies#greedy-search
- **lm-07** (fine-tuning): cited [2], gold tasks_language_modeling#preprocess
- **peft-03** (fine-tuning): cited [4], gold peft#add-an-adapter

## Format issues

None.

## Method notes

- Retrieval is the untouched BM25 baseline; this run isolates generation behaviour.
- A forced abstention means the parser refused the output (no valid citation, an invented
  URL, or a generation error), not that the model chose to abstain.
- Any model-written `Sources:` block or URL is discarded; the final source lines are
  rendered from the retrieved metadata, so the model cannot introduce a URL.
- These are automated contract metrics, not a RAG quality claim; support and
  factual correctness stay in the human label sheet.


## 7. Compare with the untuned baseline


In [15]:
import json
from pathlib import Path

base = json.loads(Path("results/answers_baseline.json").read_text())["overall"]
tuned = json.loads(Path("results/answers_tuned.json").read_text())["overall"]
keys = [
    "behavior_accuracy", "answered_rate", "grounded_answer_rate",
    "format_rate", "citation_hit_rate", "model_abstain_rate_on_negative",
    "delivered_abstention_rate_on_negative", "false_abstain_rate_on_answerable",
    "median_latency_s",
]
print(f"{'metric':42s} {'base':>8s} {'tuned':>8s}")
for key in keys:
    print(f"{key:42s} {str(base.get(key)):>8s} {str(tuned.get(key)):>8s}")


metric                                         base    tuned
behavior_accuracy                             0.116     0.93
answered_rate                                 0.023    0.628
grounded_answer_rate                            0.0    0.821
format_rate                                   0.093      1.0
citation_hit_rate                               0.0    0.821
model_abstain_rate_on_negative                0.267    0.933
delivered_abstention_rate_on_negative           1.0    0.933
false_abstain_rate_on_answerable              0.964    0.071
median_latency_s                               3.67     5.36


## 8. Checkpoint adapter and results to Drive (optional)


In [16]:
import shutil
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("Drive not mounted; skipping checkpoint:", exc)
else:
    dest = Path("/content/drive/MyDrive/transformers-docs-copilot-results")
    dest.mkdir(parents=True, exist_ok=True)
    shutil.copytree("artifacts/adapter", dest / "adapter", dirs_exist_ok=True)
    for name in ("answers_tuned.md", "answers_tuned.json", "label_sheet_tuned.csv"):
        shutil.copy2(Path("results") / name, dest / name)
    print("saved adapter and tuned results to", dest)


Mounted at /content/drive
saved adapter and tuned results to /content/drive/MyDrive/transformers-docs-copilot-results


## 9. Next steps

1. Paste the `overall` JSON and the baseline-vs-tuned table into the chat.
2. If format/abstention improved, download `label_sheet_tuned.csv` and fill the human
   support/correctness columns.
3. If format is still near zero, the 1.5B cannot learn the contract from this data;
   switch to the 3B/4B base before spending more time on prompt variants.
